# Notebook 05 — Build Monthly News Features

## Goal
Convert the approved cleaned news articles into monthly numeric features (article counts per query group).
These features represent the **information signal available at each forecast date**.

**Features created:**
- `news_volume` — total articles
- `layoff_news_count` — layoffs query group
- `hiring_news_count` — hiring query group
- `unemployment_news_count` — unemployment query group
- `job_openings_news_count` — job_openings query group
- `wage_news_count` — wages query group
- `uncertainty_news_count` — uncertainty query group

---

## What can go wrong
- Months with zero articles for a group will get `0`, not NaN — this is correct
- COVID months will have extreme layoff spikes — this is real, do not cap
- GDELT coverage before 2014 may be thinner — check early years
- Check that article_month column exists in the cleaned file

---

## What you must inspect
- Do COVID months (2020-03 to 2020-05) show layoff spikes?
- Are there months with zero articles across all groups?
- Do hiring counts peak in expansion periods?
- Are any counts dominated by duplicates?

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

CLEAN_NEWS_PATH = DRIVE_ROOT / 'data' / 'interim' / 'news' / 'clean_news_approved.parquet'

approval_04 = DRIVE_ROOT / 'approvals' / '04_news_cleaning_approved.json'
if not approval_04.exists():
    raise FileNotFoundError('STOP: Notebook 04 not approved.')
with open(approval_04) as f:
    ap = json.load(f)
assert ap['approved'], 'Notebook 04 not approved.'

news = pd.read_parquet(CLEAN_NEWS_PATH)
print(f'Cleaned news loaded: {news.shape}')
print(f'Columns: {list(news.columns)}')
print(f'article_month range: {news["article_month"].min()} to {news["article_month"].max()}')
print(f'Query groups: {news["query_group"].unique()}')

## Build monthly count features

In [ ]:
QUERY_GROUP_TO_FEATURE = {
    'layoffs': 'layoff_news_count',
    'hiring': 'hiring_news_count',
    'unemployment': 'unemployment_news_count',
    'job_openings': 'job_openings_news_count',
    'wages': 'wage_news_count',
    'uncertainty': 'uncertainty_news_count',
}

# Total volume
total_monthly = news.groupby('article_month').size().reset_index(name='news_volume')

# Per-group counts
group_monthly = news.groupby(['article_month', 'query_group']).size().unstack(fill_value=0)
group_monthly.columns = [QUERY_GROUP_TO_FEATURE.get(c, f'{c}_news_count') for c in group_monthly.columns]
group_monthly = group_monthly.reset_index()

# Merge
features_df = pd.merge(total_monthly, group_monthly, on='article_month', how='outer')
features_df = features_df.fillna(0).sort_values('article_month').reset_index(drop=True)

# Ensure all feature columns exist
for feat in QUERY_GROUP_TO_FEATURE.values():
    if feat not in features_df.columns:
        features_df[feat] = 0

print(f'News features shape: {features_df.shape}')
print(f'Columns: {list(features_df.columns)}')
print()
print(features_df.head(12).to_string(index=False))

## Diagnostic tables

In [ ]:
print('=== Feature Summary ===')
print(features_df.describe().to_string())
print()

zero_news_months = features_df[features_df['news_volume'] == 0]
print(f'Months with zero total articles: {len(zero_news_months)}')
if len(zero_news_months) > 0:
    print(zero_news_months['article_month'].tolist())
print()

print('=== Top 10 months by layoff_news_count ===')
print(features_df.nlargest(10, 'layoff_news_count')[['article_month', 'layoff_news_count', 'news_volume']].to_string(index=False))
print()

print('=== Top 10 months by hiring_news_count ===')
print(features_df.nlargest(10, 'hiring_news_count')[['article_month', 'hiring_news_count', 'news_volume']].to_string(index=False))
print()

print('=== Top 10 months by uncertainty_news_count ===')
print(features_df.nlargest(10, 'uncertainty_news_count')[['article_month', 'uncertainty_news_count', 'news_volume']].to_string(index=False))

In [ ]:
months_idx = range(len(features_df))

fig, axes = plt.subplots(4, 1, figsize=(14, 14))

axes[0].plot(months_idx, features_df['news_volume'], color='steelblue')
axes[0].set_title('Total News Volume Over Time')
axes[0].set_ylabel('Article count')
axes[0].grid(True, alpha=0.3)

axes[1].plot(months_idx, features_df['layoff_news_count'], color='red', label='Layoffs')
axes[1].plot(months_idx, features_df['hiring_news_count'], color='green', label='Hiring')
axes[1].set_title('Layoff vs Hiring News Count')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(months_idx, features_df['uncertainty_news_count'], color='orange')
axes[2].set_title('Uncertainty News Count Over Time')
axes[2].set_ylabel('Count')
axes[2].grid(True, alpha=0.3)

axes[3].plot(months_idx, features_df['unemployment_news_count'], color='purple', label='Unemployment')
axes[3].plot(months_idx, features_df['job_openings_news_count'], color='teal', label='Job openings')
axes[3].set_title('Unemployment vs Job Openings News Count')
axes[3].set_ylabel('Count')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'outputs' / 'figures' / 'news_features_over_time.png'), dpi=100)
plt.show()
print('Plot saved.')

## Assertion checks

In [ ]:
FEATURE_COLS = ['news_volume', 'layoff_news_count', 'hiring_news_count',
                'unemployment_news_count', 'job_openings_news_count',
                'wage_news_count', 'uncertainty_news_count']

assert 'article_month' in features_df.columns, 'Missing article_month column'
print('  article_month column present.')

for col in FEATURE_COLS:
    assert col in features_df.columns, f'Missing feature column: {col}'
    assert pd.api.types.is_numeric_dtype(features_df[col]), f'{col} is not numeric'
    assert (features_df[col] >= 0).all(), f'{col} has negative values'
print(f'  All {len(FEATURE_COLS)} feature columns present, numeric, and non-negative.')

assert features_df['article_month'].nunique() == len(features_df), 'Duplicate article_month values'
print('  article_month is unique key.')

# Check COVID spike exists in layoff count
if '2020-04' in features_df['article_month'].values:
    covid_layoffs = features_df[features_df['article_month'] == '2020-04']['layoff_news_count'].iloc[0]
    median_layoffs = features_df['layoff_news_count'].median()
    if covid_layoffs > median_layoffs:
        print(f'  COVID spike confirmed: 2020-04 layoff count={covid_layoffs:.0f} vs median={median_layoffs:.0f}')
    else:
        print(f'  WARNING: No COVID spike in layoff count for 2020-04 ({covid_layoffs:.0f}). Check GDELT coverage.')

print('\nAll assertion checks passed.')

## Manual Approval Gate

**Before approving:**
1. News volume plot shows expected temporal patterns
2. COVID months (2020-03 to 2020-05) show layoff spikes
3. Hiring counts peak in expansion periods
4. No implausible zero-article months in the main sample
5. Feature distributions look reasonable

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
out_path = DRIVE_ROOT / 'data' / 'features' / 'news' / 'monthly_news_features_approved.parquet'
features_df.to_parquet(out_path, index=False)
print(f'News features saved: {out_path}')

approval = {
    'approved': True,
    'notebook': '05_build_news_features.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(CLEAN_NEWS_PATH)],
    'output_artifacts': [str(out_path)],
    'row_counts': {'monthly_news_features': int(len(features_df))},
    'warnings': [],
    'notes': ''
}
ap_path = DRIVE_ROOT / 'approvals' / '05_news_features_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 05 complete. Proceed to 06_build_macro_features_and_target.ipynb')